VARIABLES
<br>
Outcome variables = <br>
lnall — Log of all-cause mortality<br>
lnmmr — Log of maternal mortality rate<br>
lnflupneu — Log of mortality from influenza and pneumonia<br>
lnscarfever — Log of mortality from scarlet fever<br>
lntb — Log of tuberculosis mortality<br>
<br>
Treatment variables = <br>post37 (1936 > 1 and 1936< 0)<br>
post37Xyear_c (interaction term between treatment and time trend)<br>
<br>


METHOD<br>
Stata data set is loaded: 20080229_national_data.dta, we filter the data from the years between 1925 and 1943, then we take the natural logarithm of the mortality variables to normalize the data. We create the treatment and time variables. Then we run a regression model and print the results to a text file (better visability).<br>


RESULTS<br>
The results match the original paper's results(Table 3 in the original paper, no exact results mentioned in Mora).


In [9]:
# Cross-moment functions

def compute_ratio(Z, D, deg=2):
    var_u = np.mean(Z * D)
    sign = np.sign(var_u)
    diff_normal_D = np.mean(D ** deg * Z) - deg * var_u * np.mean(D ** (deg - 1))
    diff_normal_Z = np.mean(Z ** deg * D) - deg * var_u * np.mean(Z ** (deg - 1))
    epsilon = 1e-8
    alpha_sq = diff_normal_D / (diff_normal_Z + epsilon)
    if alpha_sq < 0:
        alpha_sq = -(abs(alpha_sq) ** (1 / (deg - 1)))
    else:
        alpha_sq = alpha_sq ** (1 / (deg - 1))
    alpha_sq = abs(alpha_sq) * sign
    return alpha_sq

def get_beta(Z, D, Y, deg=2):
    denominator = 0
    while denominator == 0:
        alpha_sq = compute_ratio(Z, D, deg)
        numerator = np.mean(D * Y) - alpha_sq * np.mean(Y * Z)
        denominator = np.mean(D * D) - alpha_sq * np.mean(D * Z)
        deg += 1
    return numerator / denominator

In [10]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


# Load data

df = pd.read_stata("20080229_national_data.dta")
df = df[(df.year >= 1925) & (df.year <= 1943)].copy()


# Define policy variables

df["post37"] = (df["year"] > 1936).astype(int)
df["year_post"] = df["year"] - 1937
df.loc[df["year_post"] < 0, "year_post"] = 0
df["trend"] = df["year"]


# Define Table 4 disease structure

table4_structure = {
    "MMR": {
        "total": ["mmr"],
        "controls_total": ["tuberculosis_total", "influenza_pneumonia_total", "scarlet_fever_tot"]
    },
    "Pneumonia/Influenza": {
        "total": ["influenza_pneumonia_total"],
        "controls_total": ["mmr", "tuberculosis_total", "scarlet_fever_tot"]
    },
    "Scarlet fever": {
        "total": ["scarlet_fever_tot"],
        "controls_total": ["mmr", "tuberculosis_total", "influenza_pneumonia_total"]
    }
}


# Stack treated vs control for models

def stack_treatment_full(df, treated_var, control_vars):
    dfs = []

    # Treated
    df_treated = df[["year"] + treated_var].copy()
    df_treated.rename(columns={treated_var[0]: "mortality"}, inplace=True)
    df_treated["treated"] = 1
    dfs.append(df_treated)

    # Controls
    df_control = df[["year"] + control_vars].copy()
    df_control["mortality"] = df_control[control_vars].sum(axis=1)
    df_control = df_control[["year", "mortality"]]
    df_control["treated"] = 0
    dfs.append(df_control)

    stacked = pd.concat(dfs, ignore_index=True)
    stacked["lnmortality"] = np.log(stacked["mortality"])

    # Interaction terms for 5.3 & 5.4
    stacked["post37"] = (stacked["year"] > 1936).astype(int)
    stacked["treated_post"] = stacked["treated"] * stacked["post37"]
    stacked["treated_year"] = stacked["treated"] * stacked["year"]
    stacked["treated_year_post"] = stacked["treated"] * stacked["year"] * stacked["post37"]
    return stacked


# Regression functions

# Model (5.3): ln(M) ~ Treated*Post + Treated*Year + Treated + Year
def run_model_53(stacked):
    X = stacked[["treated_post", "treated_year", "treated", "year"]]
    X = sm.add_constant(X)
    y = stacked["lnmortality"]
    return sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":2})

# Model (5.4): ln(M) ~ Treated*Year*Post + Treated*Post + Treated*Year + Treated + Year
def run_model_54(stacked):
    X = stacked[["treated_year_post", "treated_post", "treated_year", "treated", "year"]]
    X = sm.add_constant(X)
    y = stacked["lnmortality"]
    return sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":2})


# Run regressions for all diseases

results_53 = {}
results_54 = {}

for disease, vars_dict in table4_structure.items():
    stacked_total = stack_treatment_full(df, vars_dict["total"], vars_dict["controls_total"])
    
    results_53[disease] = run_model_53(stacked_total)
    results_54[disease] = run_model_54(stacked_total)


# Function to format coefficients with stars

def add_stars(coef, se):
    t = abs(coef / se)
    if t > 2.58:
        star = "***"
    elif t > 1.96:
        star = "**"
    elif t > 1.65:
        star = "*"
    else:
        star = ""
    return f"{coef:.3f}{star} ({se:.3f})"


# Create formatted tables

# Model 5.3 table
rows_53 = ["Treated*Post-1937", "Treated*Year"]
cols = ["MMR", "Pneumonia/Influenza", "Scarlet fever"]

table_53 = pd.DataFrame(index=rows_53, columns=cols)
for disease in cols:
    model = results_53[disease]
    table_53[disease].iloc[0] = add_stars(model.params["treated_post"], model.bse["treated_post"])
    table_53[disease].iloc[1] = add_stars(model.params["treated_year"], model.bse["treated_year"])

# Model 5.4 table
rows_54 = ["Treated*Year*Post-1937", "Treated*Post-1937", "Treated*Year"]
table_54 = pd.DataFrame(index=rows_54, columns=cols)
for disease in cols:
    model = results_54[disease]
    table_54[disease].iloc[0] = add_stars(model.params["treated_year_post"], model.bse["treated_year_post"])
    table_54[disease].iloc[1] = add_stars(model.params["treated_post"], model.bse["treated_post"])
    table_54[disease].iloc[2] = add_stars(model.params["treated_year"], model.bse["treated_year"])

# Display results

print("\n=== Table: Model 5.3 ===")
print(table_53)
print("\n=== Table: Model 5.4 ===")
print(table_54)


=== Table: Model 5.3 ===
                                MMR Pneumonia/Influenza     Scarlet fever
Treated*Post-1937  -0.304** (0.128)      -0.163 (0.112)  -0.862** (0.334)
Treated*Year          0.011 (0.014)       0.021 (0.014)     0.016 (0.028)

=== Table: Model 5.4 ===
                                        MMR  Pneumonia/Influenza  \
Treated*Year*Post-1937    -0.108*** (0.008)    -0.087*** (0.023)   
Treated*Post-1937       208.543*** (14.818)  169.179*** (45.195)   
Treated*Year               0.029*** (0.006)     0.035*** (0.012)   

                              Scarlet fever  
Treated*Year*Post-1937    -0.254*** (0.030)  
Treated*Post-1937       491.178*** (57.556)  
Treated*Year               0.058*** (0.011)  


In [11]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Cross-moment functions
def compute_ratio(Z, D, deg=2):
    var_u = np.mean(Z * D)
    sign = np.sign(var_u)
    diff_normal_D = np.mean(D ** deg * Z) - deg * var_u * np.mean(D ** (deg - 1))
    diff_normal_Z = np.mean(Z ** deg * D) - deg * var_u * np.mean(Z ** (deg - 1))
    epsilon = 1e-8
    alpha_sq = diff_normal_D / (diff_normal_Z + epsilon)
    if alpha_sq < 0:
        alpha_sq = -(abs(alpha_sq) ** (1 / (deg - 1)))
    else:
        alpha_sq = alpha_sq ** (1 / (deg - 1))
    alpha_sq = abs(alpha_sq) * sign
    return alpha_sq

def get_beta(Z, D, Y, deg=2):
    denominator = 0
    while denominator == 0:
        alpha_sq = compute_ratio(Z, D, deg)
        numerator = np.mean(D * Y) - alpha_sq * np.mean(Y * Z)
        denominator = np.mean(D * D) - alpha_sq * np.mean(D * Z)
        deg += 1
    return numerator / denominator

# List of diseases
disease_list = [
    ("MMR", ["mmr"], ["tuberculosis_total", "influenza_pneumonia_total", "scarlet_fever_tot"]),
    ("Pneumonia/Influenza", ["influenza_pneumonia_total"], ["mmr", "tuberculosis_total", "scarlet_fever_tot"]),
    ("Scarlet fever", ["scarlet_fever_tot"], ["mmr", "tuberculosis_total", "influenza_pneumonia_total"])
]

betas_results = {}

for disease_name, treated_var, control_vars in disease_list:
    # Stack treated and control for this disease 
    stacked = stack_treatment_full(df, treated_var, control_vars)

    # Split pre/post periods 
    pre_mask = stacked["year"] <= 1936
    post_mask = stacked["year"] > 1936
    data_pre = stacked[pre_mask].copy()
    data_post = stacked[post_mask].copy()

    # Stack pre/post for cross-moment 
    Y = np.concatenate([data_pre["lnmortality"].to_numpy(), data_post["lnmortality"].to_numpy()]).astype(np.float64)
    D = np.concatenate([data_pre["treated_post"].to_numpy(), data_post["treated_post"].to_numpy()]).astype(np.float64)
    
    # Z = instrument: treated*year*post in pre-period, zeros in post
    Z = np.concatenate([data_pre["treated_year_post"].to_numpy(), np.zeros(len(data_post))]).astype(np.float64)

    #  Center variables 
    Y -= np.mean(Y)
    D -= np.mean(D)
    Z -= np.mean(Z)

    deg = 2
    beta = get_beta(Z, D, Y, deg)
    print(f"Beta estimated by cross-moment method: {beta}")

Beta estimated by cross-moment method: 0.1880776717366161
Beta estimated by cross-moment method: -1.457645222888015
Beta estimated by cross-moment method: -4.841054608847964


In [12]:
# Compare cross-moment beta with DiD Treated*Post-1937

cross_moment_results = {}

for disease, vars_dict in table4_structure.items():
    # Stack treated + controls
    stacked_total = stack_treatment_full(df, vars_dict["total"], vars_dict["controls_total"])
    
    # Split pre/post
    data_pre = stacked_total[stacked_total["year"] <= 1936]
    data_post = stacked_total[stacked_total["year"] > 1936]
    
    # Stack pre/post for cross-moment
    Y = np.concatenate([data_pre["lnmortality"].to_numpy(), data_post["lnmortality"].to_numpy()]).astype(np.float64)
    D = np.concatenate([data_pre["treated_post"].to_numpy(), data_post["treated_post"].to_numpy()]).astype(np.float64)
    
    # Z = instrument: treated*year*post in pre-period, zeros in post
    Z = np.concatenate([data_pre["treated_year_post"].to_numpy(), np.zeros(len(data_post))]).astype(np.float64)
    
    # Center variables
    Y -= np.mean(Y)
    D -= np.mean(D)
    Z -= np.mean(Z)
    
    # Estimate beta via cross-moment
    beta_cm = get_beta(Z, D, Y, deg=2)
    cross_moment_results[disease] = beta_cm

# Print comparison table

print(f"{'Disease':<20} {'DiD Treated*Post-1937':>30} {'Cross-moment beta':>25}")

for disease in table4_structure.keys():
    did_beta = results_53[disease].params["treated_post"]
    did_se = results_53[disease].bse["treated_post"]
    
    cm_beta = cross_moment_results[disease]
    
    print(f"{disease:<20} "
          f"{did_beta:>10.4f} ({did_se:.4f})"
          f"{cm_beta:>20.4f}")

Disease                       DiD Treated*Post-1937         Cross-moment beta
MMR                     -0.3039 (0.1277)              0.1881
Pneumonia/Influenza     -0.1631 (0.1122)             -1.4576
Scarlet fever           -0.8618 (0.3343)             -4.8411
